# Training Series: Introduction to eXtended Discontinuous Galerkin Methods for Multi-Phase Flow Problems <br> - Hands-On Worksheets

In [ ]:
#r "../BoSSSpad/BoSSSpad.dll"
//#r "C:\Program Files (x86)\FDY\BoSSS\bin\Release\net6.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Platform.LinAlg;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

## Hands-On 2: Scalar Transport Equation and Numerical Fluxes

### Problem statement

As an examplary problem, we consider the scalar transport equation
>$$
>\frac{\delta c}{\delta t} + \nabla \cdot (\vec{u} c) = 0
>$$
in the domain $\Omega = [-1, 1] \times [-1, 1]$, where the concentration $c = c(x,y,t) \in \mathbb{R}$ is unknown and the velocity field is given by
>$$
>\vec{u} = \begin{pmatrix}
>y\\-x
>\end{pmatrix}
>$$
The analytical solution for the problem above is given by 
>$$
>c_\text{Exact}(x,y,t) = \cos(\cos(t) x - \sin(t) y) \quad \text{ for } (x,y) \in \Omega,
>$$
which can be used to describe the initial and boundary conditions. 

### Projection and visualization

In this first section we get to know the plotting tool **Tecplot**, 
which generates .plt-files of our **DGFields**. 
Previously, we define the exact solution $c_{Exact}(x,y,t)$ and 
the scalar components of the velocity field $\vec{u}$ as functions:

In [ ]:
public static class ExactSol {
    public static double c(double[] X, double t) => Math.Cos(Math.Cos(t)*X[0] - Math.Sin(t)*X[1]);
}

In [ ]:
public static class VelField {
    public static double u(double[] X) => X[1];
    // == TODO == v-velocity

}

Next, we need to construct the computational domain, i.e a unit square with one cell.


In [ ]:
double[] nodes = GenericBlas.Linspace(-1.0, 1.0, 2); 
GridCommons grid = Grid2D.Cartesian2DGrid(nodes, nodes); 

We instantiate the **SinglePhaseField** *ch* with a **Basis** of DG-degreee of 2. Then we can project the initial value $c(x,y,0.0)$ onto *ch*.


In [ ]:
int dgDegree = 2;  
Basis basis = new Basis(grid, dgDegree);  
SinglePhaseField ch = new SinglePhaseField(basis, "ch");  
ch.ProjectField(X => ExactSol.c(X, 0.0));

Now, we can export the initial projection in our **Tecplot** format.

In [ ]:
using BoSSS.Solution.Tecplot;

One important parameter for visualization is **superSampling**. It is essential for higher order methods since almost all
plotting tools work with piecewise linear interpolations of the data in the vertices. For our
case, the plot with **superSampling**=0 would just show a constant value! By increasing the
rate of the **superSampling**, we provide more sampling points for the plot tool.

- This has nothing to do with the computation! Only required for visualization!
- The number of sampling points grows exponentially with the value of
**superSampling**. Never use a value above 5 or 6!


In [ ]:
uint superSampling = 0;  
Tecplot tecplot    = new Tecplot(grid.GridData, superSampling);
tecplot.PlotFields( 
   "plots/plot_tutorial4_superSampling0", 
   0.0, 
   ch);

In [ ]:
superSampling = 3;  
tecplot    = new Tecplot(grid.GridData, superSampling); 
tecplot.PlotFields( 
   "plots/plot_tutorial4_superSampling3", 
   0.0, 
   ch);

There should now be two plot-files in the folder plots in your current directory. Those can be opened by any standard viewer for .plt-files.

### Definition of the numerical fluxes 

Before implementing the **ScalarTransportFlux**, we define two different fluxes as functions in terms of the parameters **Uin**, **Uout**, the normal vector **n** and the
**velocityVector**. Recalling the Upwind flux, the corresponding flux function is defined as

In [ ]:
using BoSSS.Platform.LinAlg;

Func<double, double, Vector, Vector, double> upwindFlux =     
    delegate(double Uin, double Uout, Vector n, Vector velocityVector) {     
 
        if (velocityVector * n > 0) {     
            return (velocityVector * Uin) * n;     
        } else {     
           return (velocityVector * Uout) * n;     
        }     
    };

The second flux we are considering now, is the Lax-Friedrichs flux
$$
\hat{f}(c_h^-, c_h^+, \vec{u} \cdot \vec{n}) = \{ \vec{u} c_h \} \cdot \vec{n} + \frac{C}{2} \llbracket {c_h} \rrbracket.
$$
The constant $C \in \mathbb{R}^+$ has to be sufficiently large in order to guarantee the stability of the 
numerical scheme. We choose 
$$
C = \max \vert{\vec{u} \cdot \vec{n}}\vert = 1
$$
for the given problem.

In [ ]:
double C = 0.3;     
 
Func<double, double, Vector, Vector, double> laxFriedrichsFlux =     
    delegate(double Uin, double Uout, Vector n, Vector velocityVector) {     
 
        // == TODO == return the Lax-Friedrichs flux
        return 0.0;  
    };

For the implementation of the **ScalarTransportFlux** we want to generate 
instances by the definition of the **numericalFlux** and **boundaryCondition**. 

Therefore a new constructor is implemented 
and the implementation of **InnerEdgeFlux** is rewritten in terms of the **numericalFlux** and **boundaryCondition**.

In [ ]:
class ScalarTransportFlux : NonlinearFlux {     
 
    private Func<double, double, Vector, Vector, double> numericalFlux;     

    private Func<double[], double, double> boundaryCondition;
 
    // Provides instances of this class with a specific flux implementation     
    public ScalarTransportFlux(     
        Func<double, double, Vector, Vector, double> numericalFlux,
        Func<double[], double, double> boundaryCondition) {     
 
        this.numericalFlux = numericalFlux;  
        this.boundaryCondition = boundaryCondition;   
    }     
 
    public override IList<string> ArgumentOrdering {     
        get { return new string[] { "ch" }; }     
    }     
 
    protected override void Flux(     
        double time, double[] x, double[] U, double[] output) {     
 
        // == TODO == define the flux output vector 


    }     
 
    // Makes use of the flux implementation supplied in the constructor     
    protected override double InnerEdgeFlux(     
        double time, double[] x, double[] normal,      
        double[] Uin, double[] Uout, int jEdge) {     
 
        Vector n              = new Vector(normal);     
        Vector velocityVector = new Vector(VelField.u(x), VelField.v(x));     
 
        return numericalFlux(Uin[0], Uout[0], n, velocityVector);     
    }     
 
    protected override double BorderEdgeFlux(     
        double time, double[] x, double[] normal, byte EdgeTag,     
        double[] Uin, int jEdge) {     
 
        // == TODO == define the BorderEdgeFlux in terms of the InnerEdgeFlux. Hint. enforce a Dirichlet B.C. for the exact solution     
        double[] Uout = 0.0;     
 
        return 0.0;    
    }     
}

For both numerical fluxes we define a **DifferentialOperator** that uses the corresponding flux to compute div(...)

In [ ]:
var upwindOperator = new DifferentialOperator(     
    new string[] { "ch" },     
    new string[] { "div" },     
    QuadOrderFunc.NonLinear(2));

In [ ]:
upwindOperator.EquationComponents["div"].Add(     
    new ScalarTransportFlux(upwindFlux, ExactSol.c));

In [ ]:
upwindOperator.Commit();

In [ ]:
var laxFriedrichsOperator = new DifferentialOperator(     
    new string[] { "ch" },     
    new string[] { "div" },     
    QuadOrderFunc.NonLinear(2));

In [ ]:
laxFriedrichsOperator.EquationComponents["div"].Add(     
    new ScalarTransportFlux(laxFriedrichsFlux, ExactSol.c));

In [ ]:
laxFriedrichsOperator.Commit();

### Performing timestepping

We want to compute the evolution of the concentration $c(x,y,t)$ for the given velocity field.

In [ ]:
int numCells = 8;
double[] nodes = GenericBlas.Linspace(-1.0, 1.0, numCells + 1);     
GridCommons grid = Grid2D.Cartesian2DGrid(nodes, nodes); 

In [ ]:
int degree = 3;
Basis basis = new Basis(grid, degree);     
SinglePhaseField ch = new SinglePhaseField(basis, "ch_" + degree);     
ch.ProjectField(X => ExactSol.c(X, 0.0));  

For the timestepping we are using the classical fourth order Runge-Kutta scheme. 

In [ ]:
using BoSSS.Solution.Timestepping;

We consider a simulation time up to a half revolution.

In [ ]:
double endTime        = Math.PI;     
int numberOfTimesteps = 100;     
double dt             = endTime / numberOfTimesteps;

In [ ]:
// Instantiate the timestepper (classical Runge-Kutta scheme)     
// for the evolution of the concentration c with Upwind flux     
RungeKutta timeStepper = new RungeKutta(     
    RungeKutta.RungeKuttaSchemes.RungeKutta1901,     
    upwindOperator, ch);   

// plot initial solution
Tecplot tecplot = new Tecplot(grid.GridData, 3);
tecplot.PlotFields("plots/plot_HandsOn2_Timestepping_0", 0.0, ch);    
 
// Integrate in time for each timestep     
for (int k = 1; k <= numberOfTimesteps; k++) {     
    timeStepper.Perform(dt);   

    // plot current solution
    tecplot.PlotFields("plots/plot_HandsOn2_Timestepping_" + k, k * dt, ch);  
} 

There should now be plot-files from 0 t0 100 in the folder plots in your current directory.

### Convergence study

In the following a convergence study for the discretization error will be performed on grids 
with $2\times2$, $4\times4$, $8\times8$ and $16\times16$ cells.

The error at $t = \pi /4$
will be investigated for the polynomial degrees $p = 0, \ldots, 3$ of the ansatzfunctions. 

The study will be done for both numerical fluxes.

We start by defining a series of nested grids for the convergence study

In [ ]:
int[] numbersOfCells = new int[] { 2, 4, 8, 16 };

In [ ]:
GridData[] grids = new GridData[numbersOfCells.Length];

In [ ]:
for (int i = 0; i < numbersOfCells.Length; i++) {     
 
    double[] nodes = GenericBlas.Linspace(-1.0, 1.0, numbersOfCells[i] + 1);     
 
    GridCommons grid = Grid2D.Cartesian2DGrid(nodes, nodes);     
 
    grids[i] = new GridData(grid);     
}

Then, a **SinglePhaseField** is defined for each polynomial degree on each grid.
The initial value is projected on each field.

In [ ]:
int[] dgDegrees = new int[] { 0, 1, 2, 3 };

In [ ]:
SinglePhaseField[,] fields =      
    new SinglePhaseField[dgDegrees.Length, numbersOfCells.Length];

In [ ]:
for (int i = 0; i < dgDegrees.Length; i++) {     
    for (int j = 0; j < numbersOfCells.Length; j++) {     
 
        Basis basis = new Basis(grids[j], dgDegrees[i]);     
        fields[i, j] = new SinglePhaseField(     
                              basis, "ch_" + dgDegrees[i] + "_" + grids[j]);     
        fields[i, j].ProjectField(X => ExactSol.c(X, 0.0));     
    }     
}

Since the error at a fourth revolution is considered, the initial concentration has to be simulated until 
$\textbf{endTime} = \pi / 4$. 

The timestep size should be chosen small enough in order to reduce the **temporal** error,
so that the **spatial** error dominates for the convergence study.

In [ ]:
double endTime        = Math.PI / 4.0;     
int numberOfTimesteps = 100;     
double dt             = endTime / numberOfTimesteps;

A **MultidimensionalArray** is a special double array that enables **shallow** resizing and data
extraction. For example, sub-arrays can be extracted without needing to copy the entries of
the **MultidimensionalArray**. BoSSS makes extensive use of this class because this is crucial
for the performance.
Here, we create a three-dimensional array, where the first index corresponds to the flux
function, the second to the DG degree and the third to the number of cells.

In [ ]:
MultidimensionalArray errors =      
    MultidimensionalArray.Create(2, dgDegrees.Length, grids.Length);

In [ ]:
for (int i = 0; i < dgDegrees.Length; i++) {     
    for (int j = 0; j < numbersOfCells.Length; j++) {     
 
        // Clones an object and casts it to the type of the original object  
        // at the same time.  
        SinglePhaseField c = fields[i, j].CloneAs();     
 
        // Instantiate the timestepper (classical Runge-Kutta scheme)     
        // for the evolution of the concentration c with Upwind flux     
        RungeKutta timeStepper = new RungeKutta(     
            RungeKutta.RungeKuttaSchemes.RungeKutta1901,     
            upwindOperator, c);     
 
        // Integrate in time for each timestep     
        for (int k = 0; k < numberOfTimesteps; k++) {     
            timeStepper.Perform(dt);     
        }     
 
        // Compute the error w.r.t. analytical solution     
        errors[0, i, j] = c.L2Error(X => ExactSol.c(X, endTime));     
 
        // == TODO == Simulate with the Lax-Friedrichs flux     

    }     
}

### Plotting results

For the convergence plots the error is plotted over the mesh size, 
where the coarsest grid size is defined as size = 1.

In [ ]:
var sizes = numbersOfCells.Select(s => 2.0 / s).ToArray();

First, we are looking at the convergence plot of the Upwind flux. 
Therefore, an instance of Gnuplot has to be generated.

In [ ]:
var gpUpwind = new Gnuplot();     
gpUpwind.SetYRange(Math.Pow(10,-7), Math.Pow(10,0));

In [ ]:
for (int i = 0; i < dgDegrees.Length; i++) {     
 
    /// Here, we use the shallow extraction features of    
    /// \code{MultidimensionalArray} mentioned above.   
    /// The command \code{ExtractSubArrayShallow(0, i, -1)}    
    /// is roughly equivalent to writing "errors[0, i, :]" in Matlab   
    var errorsForDegree = errors.ExtractSubArrayShallow(0, i, -1).To1DArray();     
 
    // some formatting of the plot data for the convergence study      
    PlotFormat format = new PlotFormat(lineColor: (LineColors)(i+1),     
                                       Style: Styles.LinesPoints,       
                                       pointType: PointTypes.Diamond,       
                                       pointSize: 2.0);     
 
    // Create log-log plot mesh size vs. error for dgDegrees[i]    
    gpUpwind.PlotLogXLogY(sizes, errorsForDegree,      
                          "Upwind, order " + dgDegrees[i], format);     
}

In [ ]:
gpUpwind.PlotNow() // perform the plotting

The convergence plot is also done for the Lax-Friedrichs flux.

In [ ]:
var gpLaxFriedrichs = new Gnuplot();     
gpLaxFriedrichs.SetYRange(Math.Pow(10,-7), Math.Pow(10,0));

In [ ]:
for (int i = 0; i < dgDegrees.Length; i++) {     
 
    var errorsForDegree = errors.ExtractSubArrayShallow(1, i, -1).To1DArray();     
 
    PlotFormat format = new PlotFormat(lineColor: (LineColors)(i+1),     
                                       Style: Styles.LinesPoints,       
                                       pointType: PointTypes.Diamond,       
                                       pointSize: 2.0);     
 
    gpLaxFriedrichs.PlotLogXLogY(sizes, errorsForDegree,      
                                 "LaxFriedrichs, order " + dgDegrees[i],      
                                 format);     
}

In [ ]:
gpLaxFriedrichs.PlotNow()

### Linear regression by ordinary least squares

For the determination of the experimental order of convergence (EOC) the linear regression is used 
to compute the slope of the line in the log-log plot. In the following, the ordinary least squares method 
is implemented to estimate the regression coefficient of the slope for given sets of x- and y-values.

In [ ]:
Func<double[], double[], double> slope =      
    delegate(double[] xValues, double[] yValues) {     
 
    if (xValues.Length != yValues.Length) {     
        throw new ArgumentException();     
    }     
 
    xValues = xValues.Select(s => Math.Log10(s)).ToArray();     
    yValues = yValues.Select(s => Math.Log10(s)).ToArray();     
 
    double xAverage = xValues.Sum() / xValues.Length;     
    double yAverage = yValues.Sum() / yValues.Length;     
 
    double v1 = 0.0;     
    double v2 = 0.0;     
 
    // Computation of the regression coefficient for the slope     
    for (int i = 0; i < yValues.Length; i++) {     
        v1 += (xValues[i] - xAverage) * (yValues[i] - yAverage);     
        v2 += Math.Pow(xValues[i] - xAverage, 2);     
    }     
    return v1 / v2;     
};

Verification that the slope is computed correctly     

In [ ]:
Math.Abs(slope(sizes, new double[] { 64.0, 16.0, 4.0, 1.0 }) - 2.0) < 1e-14

The experimental orders of convergence (EOC) are computed for both fluxes 
for each polynomial degree of the ansatzfunctions

In [ ]:
for (int i = 0; i < dgDegrees.Length; i++) {     
 
    Console.WriteLine();     
 
    var errorsForDegree = errors.ExtractSubArrayShallow(0, i, -1).To1DArray();     
 
    Console.WriteLine(     
        "Upwind flux, order {0}:\t\t {1:F2}", i,      
        slope(sizes, errorsForDegree));     
 
    errorsForDegree = errors.ExtractSubArrayShallow(1, i, -1).To1DArray();     
 
    Console.WriteLine(     
        "Lax-Friedrichs flux, order {0}:\t {1:F2}", i,     
        slope(sizes, errorsForDegree));     
}